<a href="https://colab.research.google.com/github/RohanDevre36/Minimum-wage-causal-analysis/blob/main/01_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Minimum_Wage_Causal_Analysis"
)

folders = ["data_raw", "data_processed", "notebooks", "outputs"]

for folder in folders:
    path = PROJECT_DIR / folder
    status = "Found" if path.is_dir() else "Missing"
    print(f"{folder}: {status}")

data_raw: Found
data_processed: Found
notebooks: Found
outputs: Found


# Causal Impact of State Minimum Wage Increases
## Employment and Earnings in the United States

### Research question
How do state minimum wage increases affect employment and average
weekly wages in food services and retail?

### Project objective
Estimate how these outcomes changed after a minimum wage increase,
compared with what would likely have happened without the policy.

### Planned approach
- Collect and combine public government datasets.
- Explore trends and assess suitable comparison states.
- Apply Difference-in-Differences and an event study.
- Evaluate assumptions, uncertainty, and sensitivity to modeling choices.
- Communicate findings and policy implications.

### Scope
The policy episode, comparison states, and study period will be
selected after checking policy histories and data availability.

### Project status
Project folders created and Google Drive connection verified.
Data collection has not yet started.

In [6]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels
import requests

packages = {
    "Python": sys.version.split()[0],
    "pandas": pd.__version__,
    "NumPy": np.__version__,
    "Matplotlib": matplotlib.__version__,
    "Seaborn": sns.__version__,
    "SciPy": scipy.__version__,
    "statsmodels": statsmodels.__version__,
    "requests": requests.__version__,
}

for name, version in packages.items():
    print(f"{name}: {version}")

print("\nPackage check completed successfully.")

Python: 3.13.16
pandas: 2.2.3
NumPy: 2.1.3
Matplotlib: 3.10.0
Seaborn: 0.13.2
SciPy: 1.16.3
statsmodels: 0.15.0
requests: 2.32.4

Package check completed successfully.


In [7]:
from datetime import datetime, timezone

# Record the packages used in this notebook.
requirements = "\n".join(
    f"{name}=={version}"
    for name, version in packages.items()
    if name != "Python"
)

requirements_path = PROJECT_DIR / "requirements.txt"
requirements_path.write_text(requirements + "\n", encoding="utf-8")

# Record Python version and the date separately.
environment_path = PROJECT_DIR / "environment.txt"
environment_path.write_text(
    f"Python: {packages['Python']}\n"
    f"Recorded (UTC): {datetime.now(timezone.utc).isoformat()}\n",
    encoding="utf-8"
)

print("Saved requirements.txt and environment.txt")
print(f"Project folder: {PROJECT_DIR}")

Saved requirements.txt and environment.txt
Project folder: /content/drive/MyDrive/Minimum_Wage_Causal_Analysis


## Data Collection Plan

We will combine policy information with state-level labor market data.

| Source | Data needed | Purpose |
|---|---|---|
| US Department of Labor and state labor agencies | Minimum wage rates and effective dates | Identify policy changes |
| BLS Quarterly Census of Employment and Wages (QCEW) | Employment and average weekly wages in food services and retail | Measure industry outcomes |
| BLS Local Area Unemployment Statistics (LAUS) | State unemployment rates | Describe broader labor market conditions |

### Planned data structure
Each row will represent one state, industry, and time period.
We will choose the time frequency after checking data availability.

### Important measurement distinction
Average weekly wages are not hourly wage rates. They can change
because of hours worked or changes in the composition of employees,
as well as changes in pay.

### Study selection
We will verify policy dates and data coverage before selecting the
treated states, comparison states, and study period. Comparison
selection will use policy histories and pre-policy information,
rather than the size or significance of estimated effects.

In [8]:
from io import BytesIO

url = "https://data.bls.gov/cew/data/api/2019/1/industry/722.csv"

response = requests.get(url, timeout=60)
response.raise_for_status()

sample = pd.read_csv(
    BytesIO(response.content),
    dtype={"area_fips": str, "industry_code": str}
)

# Check the response before saving it.
required_columns = {
    "area_fips", "industry_code", "own_code",
    "month1_emplvl", "avg_wkly_wage"
}

if not required_columns.issubset(sample.columns):
    raise ValueError("The download did not contain the expected BLS columns.")

raw_path = PROJECT_DIR / "data_raw" / "qcew_2019_q1_722.csv"
raw_path.write_bytes(response.content)

print(f"Downloaded {sample.shape[0]:,} rows and {sample.shape[1]} columns.")
print(f"Saved: {raw_path.name}")

display(sample.head())

Downloaded 4,152 rows and 42 columns.
Saved: qcew_2019_q1_722.csv


,area_fips,own_code,industry_code,agglvl_code,size_code,year,qtr,disclosure_code,qtrly_estabs,month1_emplvl,...,oty_month3_emplvl_chg,oty_month3_emplvl_pct_chg,oty_total_qtrly_wages_chg,oty_total_qtrly_wages_pct_chg,oty_taxable_qtrly_wages_chg,oty_taxable_qtrly_wages_pct_chg,oty_qtrly_contributions_chg,oty_qtrly_contributions_pct_chg,oty_avg_wkly_wage_chg,oty_avg_wkly_wage_pct_chg
0,01000,1,722,55,0,2019,1,NaN,6,208,...,-12,-5.5,1614,0.1,0,0.0,0,0.0,24,6.1
1,01000,3,722,55,0,2019,1,N,145,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
2,01000,5,722,55,0,2019,1,NaN,8642,163474,...,2269,1.4,30898280,4.9,22526342,4.0,25904,0.5,9,3.0
3,01001,3,722,75,0,2019,1,N,1,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
4,01001,5,722,75,0,2019,1,NaN,67,1195,...,-18,-1.5,88596,2.1,56550,1.5,-3632,-11.2,12,4.6


In [9]:
print('total rows:', sample.shape[0])
print('total columns:', sample.shape[1])

total rows: 4152
total columns: 42


In [10]:
# Before filtering, will check how BLS identifies the records.

print("ownership codes and record counts:")
print(sample["own_code"].value_counts().sort_index())

print("\nFirst 15 geographic codes:")
print(sample["area_fips"].drop_duplicates().head(15).tolist())

print("\nAvailable columns:")
print(sample.columns.tolist())



ownership codes and record counts:
own_code
1     166
2       7
3     287
5    3692
Name: count, dtype: int64

First 15 geographic codes:
['01000', '01001', '01003', '01005', '01007', '01009', '01011', '01013', '01015', '01017', '01019', '01021', '01023', '01025', '01027']

Available columns:
['area_fips', 'own_code', 'industry_code', 'agglvl_code', 'size_code', 'year', 'qtr', 'disclosure_code', 'qtrly_estabs', 'month1_emplvl', 'month2_emplvl', 'month3_emplvl', 'total_qtrly_wages', 'taxable_qtrly_wages', 'qtrly_contributions', 'avg_wkly_wage', 'lq_disclosure_code', 'lq_qtrly_estabs', 'lq_month1_emplvl', 'lq_month2_emplvl', 'lq_month3_emplvl', 'lq_total_qtrly_wages', 'lq_taxable_qtrly_wages', 'lq_qtrly_contributions', 'lq_avg_wkly_wage', 'oty_disclosure_code', 'oty_qtrly_estabs_chg', 'oty_qtrly_estabs_pct_chg', 'oty_month1_emplvl_chg', 'oty_month1_emplvl_pct_chg', 'oty_month2_emplvl_chg', 'oty_month2_emplvl_pct_chg', 'oty_month3_emplvl_chg', 'oty_month3_emplvl_pct_chg', 'oty_total_qtrly

In [14]:
# FIPS codes identify geographic areas.
# This list inclides the 50 states and washington, DC
# Keep them as strings so leading zeros, such as "01", are preserved.

state_fips = """
01 02 04 05 06
08 09 10 11 12 13
15 16 17 18 19 20 21 22
23 24 25 26 27 28 29 30
31 32 33 34 35 36 37 38
39 40 41 42 44 45 46 47
48 49 50 51 53 54 55 56
""".split()

# Convert each two-digit code into a five charater statewide code.
#Example: alabama "01" becomes "01000".
state_area_codes = [code + "000" for code in state_fips]

#select onlu the records that meet all four conditions below.
#The & symbol menas "AND": .eq() menas "equals".
state_sample = sample.loc[
    #keep only the 50 states and washington DC
    sample["area_fips"].isin(state_area_codes)
    #owenership code 5 menas private-sector emplyoers
    & sample["own_code"].eq(5)
    # Aggregation code 55 means statewide, three-digit industry records.
    & sample["agglvl_code"].eq(55)
    # Industry 722 means food services and drinking places.
    & sample["industry_code"].eq("722")

# Make an independent copy so we can work without changing sample.
].copy()

#Check how many records remain. we expect one per state/DC
print("State/DC records:", len(state_sample))

#check whether any geographic area appears more than once
# # Zero duplicates is the expected result for this single-quarter sample.
print(
    "Duplicate areas:",
    state_sample["area_fips"].duplicated().sum()
)

# Inspect disclosure flags before using employment or wage values.
# Some data may be withheld to protect employer confidentiality.
# Replace missing flags with "(blank)" only for this printed summary.
print("\nDisclosure flags:")
print(
    state_sample["disclosure_code"]
    .fillna("(blank)")
    .value_counts()
)

# Preview the geographic code, time period, monthly employment,
# and average weekly wage for the first five selected records.
display(
    state_sample[
        [
            "area_fips",
            "year",
            "qtr",
            "month1_emplvl",
            "month2_emplvl",
            "month3_emplvl",
            "avg_wkly_wage"
        ]
    ].head()
)

State/DC records: 51
Duplicate areas: 0

Disclosure flags:
disclosure_code
(blank)    51
Name: count, dtype: int64


,area_fips,year,qtr,month1_emplvl,month2_emplvl,month3_emplvl,avg_wkly_wage
2,01000,2019,1,163474,166391,168111,309
99,02000,2019,1,20023,20303,20491,416
131,04000,2019,1,236992,240813,244560,409
155,05000,2019,1,94649,95886,97053,307
235,06000,2019,1,1423433,1441824,1456022,451


In [15]:
##Calculate average monthly employment for the quarter
# These columns contain employment counts for each month of Q1.
employment_columns = [
    "month1_emplvl",
    "month2_emplvl",
    "month3_emplvl"
]

# Calculate average monthly employment during the quarter.
# skipna=False prevents calculating a partial average if a month is missing.
state_sample["avg_monthly_employment"] = (
    state_sample[employment_columns].mean(axis=1, skipna=False)
)

# Inspect missing values in the two outcomes we plan to analyze.
outcome_columns = ["avg_monthly_employment", "avg_wkly_wage"]

print("Missing values:")
print(state_sample[outcome_columns].isna().sum())

# Show the first five records with a fresh, consecutive display index.
display(
    state_sample[
        ["area_fips", "year", "qtr",
         "avg_monthly_employment", "avg_wkly_wage"]
    ].reset_index(drop=True).head()
)

Missing values:
avg_monthly_employment    0
avg_wkly_wage             0
dtype: int64


,area_fips,year,qtr,avg_monthly_employment,avg_wkly_wage
0,01000,2019,1,1.659920e+05,309
1,02000,2019,1,2.027233e+04,416
2,04000,2019,1,2.407883e+05,409
3,05000,2019,1,9.586267e+04,307
4,06000,2019,1,1.440426e+06,451


In [16]:
##Save the prepared sample
# Keep identifiers, disclosure information, and our two outcomes.
# Retain industry and ownership codes so the file describes its scope.
columns_to_save = [
    "area_fips",
    "industry_code",
    "own_code",
    "year",
    "qtr",
    "disclosure_code",
    "avg_monthly_employment",
    "avg_wkly_wage"
]

# Sort by geographic code and create consecutive row indexes.
prepared_sample = (
    state_sample[columns_to_save]
    .sort_values("area_fips")
    .reset_index(drop=True)
)

# Save the prepared data separately from the original raw download.
output_path = (
    PROJECT_DIR / "data_processed"
    / "qcew_state_private_food_services_2019_q1.csv"
)

# index=False prevents writing the DataFrame's row index into the CSV.
prepared_sample.to_csv(output_path, index=False)

print(f"Saved: {output_path.name}")
print(f"Rows: {len(prepared_sample)}")

# Format the preview for readability without changing stored values.
display(
    prepared_sample.head().style.format({
        "avg_monthly_employment": "{:,.2f}",
        "avg_wkly_wage": "${:,.0f}"
    })
)

Saved: qcew_state_private_food_services_2019_q1.csv
Rows: 51


,area_fips,industry_code,own_code,year,qtr,disclosure_code,avg_monthly_employment,avg_wkly_wage
0,01000,722,5,2019,1,nan,"165,992.00",$309
1,02000,722,5,2019,1,nan,"20,272.33",$416
2,04000,722,5,2019,1,nan,"240,788.33",$409
3,05000,722,5,2019,1,nan,"95,862.67",$307
4,06000,722,5,2019,1,nan,"1,440,426.33",$451
